# LLB RAG — Offline Study Assistant
### Agentic AI Course Project · Retrieval-Augmented Generation over your own notes

This notebook answers questions using **your** LLB study materials (all semesters),
running entirely on free, open-source models — no paid API, no key.

**The 4 stages of RAG** (each has its own section below, and each cell prints what
it produced so you can watch the pipeline work):

| Stage | What happens | Tool |
|-------|--------------|------|
| 1. Load & Chunk | read your documents, split into passages | `pypdf`, `python-docx` |
| 2. Embed | turn each passage into a meaning-vector | `sentence-transformers` |
| 3. Store & Retrieve | save vectors; find ones near your question | `chromadb` |
| 4. Generate | local LLM answers from retrieved passages | `Ollama` |

**Run order:** top to bottom. Read the note above each cell first — that's the learning.


---
# Section 0 — Connect to Google Drive & set up the project

Everything lives in `My Drive/LLB RAG`. We mount Drive so the notebook can read your
documents and *save* the database there (so it persists between Colab sessions).


### 0.1 — Mount Google Drive
A pop-up will ask permission. After this, your Drive is available at
`/content/drive/MyDrive/`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("\nDrive mounted. Your files are now under /content/drive/MyDrive/")

### 0.2 — Load settings from `config.py`
All tweakable values (paths, chunk size, models, which semesters to load) live in
`config.py` in the project root. We load it here so the rest of the notebook reads
its settings from one place.


In [ ]:
import sys, importlib

PROJECT_ROOT = "/content/drive/MyDrive/LLB RAG"
sys.path.insert(0, PROJECT_ROOT)

import config
importlib.reload(config)   # pick up edits without restarting the runtime

print("Settings loaded from config.py:")
print("  Documents folder :", config.DOCUMENTS_DIR)
print("  Vector DB folder :", config.VECTOR_DB_DIR)
print("  Semesters to load:", config.SEMESTERS_TO_LOAD)
print("  Chunk size       :", config.CHUNK_SIZE, "chars (overlap", config.CHUNK_OVERLAP, ")")
print("  Embedding model  :", config.EMBEDDING_MODEL)
print("  LLM model        :", config.LLM_MODEL)
print("  Retrieve top K   :", config.TOP_K)

### 0.3 — Scaffold the folder structure (safety net)
This recreates any missing folders inside `LLB RAG` — useful if you start fresh or a
folder got deleted. If everything already exists (from the uploaded zip), it simply
confirms and changes nothing. *This is the "self-building" part of the project.*


In [ ]:
import os

SEM_FOLDERS = ["sem1", "sem2", "sem3", "sem4", "sem5", "sem6"]
SEM2_SUBJECTS = ["contract_law_2", "constitutional_law_2", "property_law",
                 "family_law_2", "labour_law_1"]

def scaffold():
    os.makedirs(config.VECTOR_DB_DIR, exist_ok=True)
    for sem in SEM_FOLDERS:
        os.makedirs(os.path.join(config.DOCUMENTS_DIR, sem), exist_ok=True)
    for subj in SEM2_SUBJECTS:
        os.makedirs(os.path.join(config.DOCUMENTS_DIR, "sem2", subj), exist_ok=True)

scaffold()
print("Folder structure verified / created under:", PROJECT_ROOT)
for sem in SEM_FOLDERS:
    p = os.path.join(config.DOCUMENTS_DIR, sem)
    print(f"  {sem}: exists")

---
# Section 1 — Install the toolkit

Run once per Colab session (Colab forgets installs when it closes). Re-running this
is normal and is itself part of seeing how the pipeline is assembled.


In [ ]:
# Document readers + embeddings + vector database
!pip install -q sentence-transformers chromadb pypdf python-docx ollama
print("\nCore RAG libraries installed.")

### 1.1 — Install Ollama + the local LLM (for Stage 4)
Ollama runs an open-source language model **on this machine** — no external API. We
pull `llama3.2:3b` (small, ~2GB, one-time download). This is what makes the system
fully self-hosted, which is the point of the project.


In [ ]:
import subprocess, time

# Install zstd first (Ollama's installer now requires it to unpack)
!apt-get -qq install -y zstd

# Install and start the Ollama server (Linux/Colab)
!curl -fsSL https://ollama.com/install.sh | sh
subprocess.Popen(["ollama", "serve"])
time.sleep(5)

print(f"Pulling {config.LLM_MODEL} (one-time download)...")
subprocess.run(["ollama", "pull", config.LLM_MODEL])
print("\nLocal LLM ready.")

---
# Stage 1 — Load & Chunk

**Goal:** read every document in `documents/` and split it into small overlapping
passages ("chunks") that we can search.

**Why chunk?** A whole 40-page PDF is too big to hand a model and too coarse to
search precisely. Small chunks let retrieval pin-point the exact relevant passage.
**Overlap** means a sentence sitting on a boundary still appears whole in one chunk.

Each chunk remembers **where it came from** (semester, subject folder, filename) —
this powers source citations and lets you filter searches later.


In [ ]:
import os
from pypdf import PdfReader
import docx

def read_pdf(path):
    text = ""
    for page in PdfReader(path).pages:
        text += (page.extract_text() or "") + "\n"
    return text

def read_docx(path):
    return "\n".join(p.text for p in docx.Document(path).paragraphs)

def read_txt(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def read_any(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".pdf":  return read_pdf(path)
    if ext == ".docx": return read_docx(path)
    if ext == ".txt":  return read_txt(path)
    return ""

def chunk_text(text, size, overlap):
    text = " ".join(text.split())            # tidy whitespace
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap              # step back -> overlap
    return chunks

def should_load(rel_path):
    """Respect SEMESTERS_TO_LOAD from config."""
    if "all" in config.SEMESTERS_TO_LOAD:
        return True
    return any(sem in rel_path for sem in config.SEMESTERS_TO_LOAD)

# Walk the documents folder recursively
all_chunks, metadatas = [], []
files_loaded = 0

for root, _, files in os.walk(config.DOCUMENTS_DIR):
    for fn in files:
        if fn.startswith(".") or fn == ".keep.txt":
            continue
        ext = os.path.splitext(fn)[1].lower()
        if ext not in config.SUPPORTED_EXTENSIONS:
            continue
        full = os.path.join(root, fn)
        rel = os.path.relpath(full, config.DOCUMENTS_DIR)   # e.g. sem2/property_law/x.pdf
        if not should_load(rel):
            continue
        try:
            text = read_any(full)
        except Exception as e:
            print(f"  skipped {rel}: {e}")
            continue
        if not text.strip():
            continue
        files_loaded += 1
        parts = rel.split(os.sep)
        semester = parts[0] if parts else "unknown"
        subject = parts[1] if len(parts) > 2 else "general"
        for i, ch in enumerate(chunk_text(text, config.CHUNK_SIZE, config.CHUNK_OVERLAP)):
            if ch.strip():
                all_chunks.append(ch.strip())
                metadatas.append({"source": fn, "semester": semester,
                                  "subject": subject, "path": rel, "chunk_no": i})

print(f"Loaded {files_loaded} document(s) -> {len(all_chunks)} chunks total\n")
if all_chunks:
    print("Example chunk:")
    print("  from :", metadatas[0]["path"])
    print("  text :", all_chunks[0][:200], "...")
else:
    print("No documents found yet. Add files under documents/ and re-run this cell.")

---
# Stage 2 — Embed

**Goal:** turn each text chunk into an **embedding** — a list of numbers (a vector)
that represents its *meaning*. Chunks about similar ideas end up with similar vectors.
That is the mechanism that lets us search by meaning instead of exact keywords.

We use `all-MiniLM-L6-v2` (from config): free, offline, downloaded once. Each chunk
becomes a 384-number vector. The cell prints the shape so you see "text → numbers".


In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(config.EMBEDDING_MODEL)

if all_chunks:
    embeddings = embedder.encode(all_chunks, show_progress_bar=True)
    print(f"\nEmbedded {len(all_chunks)} chunks.")
    print(f"Each chunk is now a vector of {embeddings.shape[1]} numbers.")
    print("First vector (first 8 numbers):", embeddings[0][:8])
else:
    embeddings = []
    print("No chunks to embed yet — add documents first (Stage 1).")

---
# Stage 3 — Store & Retrieve

**Goal:** save the vectors in **ChromaDB** (a local vector database stored in your
Drive's `vector_db/` folder, so it survives between sessions). Then, given a question,
embed the question the same way and ask Chroma for the nearest chunks — the most
relevant passages.

### The REBUILD switch
- `REBUILD = True`  -> wipe and rebuild the database from current documents.
  Use this the **first time**, and **whenever you add new study files**.
- `REBUILD = False` -> just open the existing saved database (fast; no re-embedding).
  Use this for normal day-to-day questioning.


In [ ]:
import chromadb

REBUILD = True   # <-- set False after first successful build (and re-True when you add docs)

chroma_client = chromadb.PersistentClient(path=config.VECTOR_DB_DIR)
COLLECTION = "llb_notes"

if REBUILD:
    try:
        chroma_client.delete_collection(COLLECTION)
    except Exception:
        pass
    collection = chroma_client.create_collection(COLLECTION)
    if all_chunks:
        collection.add(
            documents=all_chunks,
            embeddings=[e.tolist() for e in embeddings],
            metadatas=metadatas,
            ids=[f"chunk_{i}" for i in range(len(all_chunks))],
        )
    print(f"Database REBUILT and saved to Drive. Stored {collection.count()} chunks.")
else:
    collection = chroma_client.get_collection(COLLECTION)
    print(f"Opened existing database. {collection.count()} chunks available.")

### 3.1 — The retrieval function
Embed the question, ask Chroma for the closest chunks, return them with their source
and a *distance* (smaller = more similar). Optionally filter by semester or subject.


In [ ]:
def retrieve(question, k=None, semester=None, subject=None):
    k = k or config.TOP_K
    q_vec = embedder.encode([question])[0].tolist()
    where = {}
    if semester: where["semester"] = semester
    if subject:  where["subject"] = subject
    res = collection.query(
        query_embeddings=[q_vec],
        n_results=k,
        where=where or None,
    )
    hits = []
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        hits.append({"text": doc, "source": meta["source"],
                     "path": meta["path"], "distance": dist})
    return hits

# Try it (works once you have documents loaded)
demo_q = "What is a contract of indemnity?"
for i, h in enumerate(retrieve(demo_q, k=3), 1):
    print(f"--- Result {i}  ({h['path']}, distance {h['distance']:.3f}) ---")
    print(h["text"][:220], "...\n")

---
# Stage 4 — Generate (the "G" in RAG)

**Goal:** hand the retrieved passages *and* the question to the local LLM, instructing
it to answer **using only that context** and to cite the source. This grounding is
what stops the model from inventing answers — it speaks from your notes.

Study the `prompt` string below: **instructions + retrieved context + question**.
That construction *is* RAG. Everything before this stage existed to fill in `context`.


In [ ]:
import ollama
import subprocess, time

def ensure_ollama(retries=3):
    """Self-healing guard: make sure the local Ollama server is running.
    In Colab the background server can die after idle time; this restarts it quietly."""
    for _ in range(retries):
        try:
            ollama.list()                      # cheap ping
            return
        except Exception:
            subprocess.Popen(["ollama", "serve"],
                             stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            time.sleep(5)
    raise RuntimeError("Ollama server could not be started. Re-run cell 1.1.")

def rag_answer(question, k=None, semester=None, subject=None):
    hits = retrieve(question, k=k, semester=semester, subject=subject)
    if not hits:
        return "No documents in the database yet. Add files and rebuild.", []

    context = "\n\n".join(f"[Source: {h['path']}]\n{h['text']}" for h in hits)

    prompt = f"""You are a study assistant for an LLB law student. Answer the
question using ONLY the context below. If the context does not contain the answer,
say you don't have that material. Be precise with section numbers and case names.
Cite the source file(s) you used.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

    ensure_ollama()                            # revive server if it died
    resp = ollama.chat(model=config.LLM_MODEL,
                       messages=[{"role": "user", "content": prompt}])
    return resp["message"]["content"], hits

answer, sources = rag_answer("What is a contract of indemnity?")
print("ANSWER:\n")
print(answer)
print("\n" + "="*60)
print("Grounded in:", ", ".join(sorted({h["path"] for h in sources})))

---
# Use it — ask anything

Change `my_question` and re-run. Optional filters let you target one semester or
subject, e.g. `rag_answer(q, subject="property_law")`.


In [ ]:
my_question = "Distinguish between sale and mortgage under the Transfer of Property Act"

answer, sources = rag_answer(my_question)
print("Q:", my_question, "\n")
print(answer)
print("\nSources:", ", ".join(sorted({h["path"] for h in sources})))

---
# Section 5 — Capstone: Interactive RAG Chatbot

So far you ask one question per cell. A real RAG *application* (the Agentathon
capstone deliverable) keeps a conversation going and has a usable interface.

This section adds three things on top of the `rag_answer()` you already built —
**without changing the offline stack**:

1. **A terminal-style chat loop** — ask many questions in a row, type `exit` to stop.
2. **Conversation memory** — follow-up questions ("what about property law?") understand
   what you asked before.
3. **A Gradio chat UI** — a real chat window, the polished capstone deliverable.

Everything still runs on your local Ollama + ChromaDB. No API, no key.


### 5.1 — A conversation-aware answer function

The `rag_answer()` from Stage 4 treats every question as brand-new. For a chatbot we
want follow-ups to make sense. So we add a light **history**: the last few
question/answer pairs are passed to the LLM as prior turns, and we use the *recent*
conversation to enrich what we retrieve.

This is the one genuinely new idea here — the Mission 5 reference chatbot is stateless;
ours remembers. Note we keep history short (last 3 turns) so the prompt stays focused.


In [ ]:
import ollama

# Conversation memory: list of (question, answer) pairs
chat_history = []
HISTORY_TURNS = 3   # how many recent turns to remember

def rag_chat(question, k=None, semester=None, subject=None):
    """RAG answer that is aware of recent conversation."""
    # Retrieve using the question (optionally enriched by the last question for context)
    search_text = question
    if chat_history:
        last_q = chat_history[-1][0]
        search_text = f"{last_q} {question}"   # helps resolve "what about X?" follow-ups
    hits = retrieve(search_text, k=k, semester=semester, subject=subject)

    if not hits:
        return "No documents in the database yet. Add files and rebuild.", []

    context = "\n\n".join(f"[Source: {h['path']}]\n{h['text']}" for h in hits)

    # Build prior-turns text from recent history
    history_text = ""
    for prev_q, prev_a in chat_history[-HISTORY_TURNS:]:
        history_text += f"\nEarlier question: {prev_q}\nEarlier answer: {prev_a}\n"

    prompt = f"""You are a study assistant for an LLB law student. Answer the
question using ONLY the context below. If the context does not contain the answer,
say you don't have that material. Be precise with section numbers and case names.
Cite the source file(s) you used. Use the earlier conversation only to understand
follow-up questions, never as a source of facts.

EARLIER CONVERSATION (for context only):
{history_text if history_text else "(none)"}

CONTEXT (your study notes — the only source of facts):
{context}

QUESTION: {question}

ANSWER:"""

    ensure_ollama()                            # revive server if it died
    resp = ollama.chat(model=config.LLM_MODEL,
                       messages=[{"role": "user", "content": prompt}])
    answer = resp["message"]["content"]

    # Save this turn to memory
    chat_history.append((question, answer))
    return answer, hits

# Quick test of memory: ask, then ask a follow-up
chat_history.clear()
a1, _ = rag_chat("What is a contract of indemnity?")
print("Q1: What is a contract of indemnity?")
print(a1[:300], "...\n")
print("(memory now holds", len(chat_history), "turn)")

### 5.2 — Terminal chat loop (Mission 5 lesson 11 style)

Run this cell and type questions in the box that appears. Type `exit` (or `quit`) to
stop. This mirrors the reference HR chatbot's loop, but on your offline stack and with
memory. Sources are shown under each answer so you can verify grounding.


In [ ]:
def chat_loop():
    print("=" * 60)
    print("LLB STUDY CHATBOT  —  type 'exit' to quit")
    print("=" * 60)
    chat_history.clear()
    while True:
        try:
            q = input("\nYou: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye!")
            break
        if q.lower() in ("exit", "quit", "q"):
            print("Goodbye! Good luck with your exams.")
            break
        if not q:
            continue
        print("...checking your notes...")
        answer, sources = rag_chat(q)
        print(f"\nAssistant: {answer}")
        if sources:
            srcs = ", ".join(sorted({h["path"] for h in sources}))
            print(f"  (sources: {srcs})")

# Run it (uncomment the line below). Left off so "Run all" reaches the Gradio UI.
# chat_loop()

### 5.3 — Gradio chat UI (Mission 5 lesson 12 style)

A proper chat window, the polished capstone deliverable. Gradio is only the *interface*
— retrieval and generation still run on your local Ollama + ChromaDB, fully offline.

When you run this, Colab prints a public link you can open in a new tab (or it appears
inline). Closing the notebook stops the app.


In [ ]:
!pip install -q gradio
import gradio as gr

def gradio_respond(message, history):
    # history uses the new 'messages' format; our chat_history handles memory
    answer, sources = rag_chat(message)
    if sources:
        srcs = ", ".join(sorted({h["path"] for h in sources}))
        answer = f"{answer}\n\n_Sources: {srcs}_"
    return answer

chat_history.clear()

# Gradio 5 needs type="messages"; Gradio 6+ removed the argument (messages is default)
extra = {"type": "messages"} if int(gr.__version__.split(".")[0]) < 6 else {}

demo = gr.ChatInterface(
    fn=gradio_respond,
    **extra,
    title="LLB Study Assistant (Offline RAG)",
    description="Ask questions about your LLB notes. Answers are grounded in your "
                "own documents and cite their source. Runs fully offline.",
    examples=[
        "What is a contract of indemnity?",
        "Distinguish between sale and mortgage under TPA",
        "What are the grounds for divorce under Muslim law?",
    ],
)

demo.launch(share=True, debug=True)

### 5.4 — What this adds for your capstone writeup

You now have a full **RAG application**, not just a pipeline:

- An **interactive interface** (terminal loop *and* a Gradio chat UI) — matching the
  Mission 5 capstone deliverable (lessons 11–12), implemented on a fully offline stack.
- **Conversation memory** — follow-up questions resolve against recent turns. The
  reference Mission 5 chatbot is stateless, so this is a genuine improvement to mention.
- **Grounded, source-cited answers** over *your own domain* (LLB notes) — exactly the
  "domain-specific RAG capstone" the Agentathon describes.

**Stack mapping vs the Mission 5 reference** (useful for your report):
- Reference uses OpenAI embeddings + GPT-4o-mini; yours uses `all-MiniLM-L6-v2` +
  local `llama3.2:3b` — same architecture, self-hosted instead of API-based.
- Reference uses FAISS; yours uses ChromaDB — both are vector databases doing
  nearest-neighbour search.
- Reference uses LangChain LCEL to chain steps; yours wires the same steps explicitly
  in Python — same data flow (retrieve → format → prompt → generate), shown directly
  so the mechanism is visible.


---
# Recap — for your project report

You built a complete **offline RAG pipeline** over your own study corpus:

1. **Load & Chunk** — recursive read of `documents/`, split into overlapping passages,
   each tagged with semester / subject / filename.
2. **Embed** — `all-MiniLM-L6-v2` maps each passage to a 384-dim meaning vector.
3. **Store & Retrieve** — ChromaDB persists vectors in Drive and does semantic
   nearest-neighbour search, with optional metadata filters.
4. **Generate** — a self-hosted `llama3.2:3b` answers grounded only in retrieved
   passages, citing sources. No external API.

**Why RAG beats asking a bare LLM:** answers come from *your* curated notes — grounded,
source-cited, and updated just by adding files (no retraining).

**Extensions worth exploring (and writing up):**
- Tune `CHUNK_SIZE` / `CHUNK_OVERLAP` and observe retrieval quality.
- Tune `TOP_K`: too low misses context, too high adds noise.
- Try a stronger embedding model (`all-mpnet-base-v2`) or LLM (`llama3.1:8b`).
- Use the metadata filters to build per-subject assistants.
- **Bridge to Agentic AI:** wrap `rag_answer()` as a *tool* an agent can call — the
  agent decides *when* to retrieve, chains it with other tools, and plans multi-step
  answers. This RAG is the retrieval backbone of that agent.
